In this notebook, we test the predict update_stats and predict method from elo_v4.py.  
We also test different K-factor and pick the one with best performance for the elo rating system.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Import professional snooker matches data form 1982 to 2025
matches = pd.read_csv('/Users/tliu/Desktop/Erdos Project/Data/matches.csv')
matches.head()

,player1,player2,score1,score2,best_of,tournament_id,date,year
0,Colin Roscoe,Jackie Rea,9,6,17,753,NaN,1982
1,Tommy Murphy,Clive Everton,9,4,17,753,NaN,1982
2,Vic Harris,Marcus Owen,9,4,17,753,NaN,1982
3,Bob Harris,Graham Cripsey,9,6,17,753,NaN,1982
4,Geoff Foulds,Matt Gibson,9,3,17,753,NaN,1982


In [3]:
# Import the elo rating system and update players elo and statistics using hitorical matches.
# For now we use 6 as our K-factor.
from elo_v4 import elo_rating
elo = elo_rating()
elo.update_stats(matches, K_factor = 6)

,elo_rating,matches_played,matches_won,matches_win_rate,frames_played,frames_won,frames_win_rate
A Barrett,997,1,0,0.000000,5,2,0.400000
A Bulajiang,1002,8,1,0.125000,48,17,0.354167
A Chan,997,2,1,0.500000,9,4,0.444444
A Coleman,994,1,0,0.000000,6,2,0.333333
A Noad,991,1,0,0.000000,3,0,0.000000
...,...,...,...,...,...,...,...
Zsolt Fenyvesi,970,4,0,0.000000,21,5,0.238095
Zulfigar Cheema,883,9,0,0.000000,53,8,0.150943
Zulfiqar Qadir,996,6,2,0.333333,35,15,0.428571
Zung Chun Hong,997,1,0,0.000000,2,0,0.000000


In [4]:
stats = elo.stats.sort_values(by = 'elo_rating', ascending = False)
stats.head(10)

,elo_rating,matches_played,matches_won,matches_win_rate,frames_played,frames_won,frames_win_rate
Judd Trump,1584,1534,1106,0.720991,10667,6449,0.604575
John Higgins,1551,1933,1363,0.705122,15024,8851,0.589124
Zhao Xintong,1549,375,223,0.594667,2525,1419,0.561980
Mark Selby,1544,1640,1121,0.683537,11668,6793,0.582191
Yan Bingtao,1525,354,232,0.655367,2479,1406,0.567164
Kyren Wilson,1521,1122,721,0.642602,7218,4082,0.565531
Ronnie O'Sullivan,1510,1555,1181,0.759486,13448,8192,0.609161
Barry Hawkins,1505,1391,870,0.625449,9636,5396,0.559983
Mark Williams,1494,1835,1201,0.654496,13500,7733,0.572815
Neil Robertson,1490,1331,883,0.663411,9729,5599,0.575496


In [5]:
#In this cell, matches history before tournament xxxx to predict the scores of matches in tournament xxxx
#We will compare our prediction with the actual scores. The mse will be recorded.

from sklearn.metrics import mean_squared_error

#Store all the tournaments in a list
tournaments = list(matches.tournament_id.unique())

#Ks represent K-factors we use
#nth means nth tournament among 1085 tournaments
Ks = [4, 6, 8, 15, 20]
nth = [750, 999, 1040, 1080]
scores1 = np.zeros((5,4))

i=0
for K in Ks:
    j=0
    for n in nth:
        past_tournaments = tournaments[:n]
        current = tournaments[n]
        past_index = matches['tournament_id'].map(lambda x: x in past_tournaments)
        

        matches_past = matches[past_index]
        matches_future = matches[matches['tournament_id'] == current]

        #Update statistics of players
        elo = elo_rating()
        elo.update_stats(matches = matches_past, K_factor = K)

        #Store the actual result in y
        matches_future['result'] = 0  
        y = np.array(matches_future['result'])

        #Compute expected win_rate
        X = matches_future[['player1', 'player2', 'best_of']]
        _, expected, _ = elo.predict(X)

        #Compute the actual frame_rate
        actual = np.array(matches_future['score1']/(matches_future['score1']+matches_future['score2']))

        #Record the accuracy score
        scores1[i,j] = mean_squared_error(expected, actual)
        
        j += 1
    i+=1




/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/820499233.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_future['result'] = 0
/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/820499233.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_future['result'] = 0
/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/820499233.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

In [6]:
scores = pd.DataFrame(scores1, index = [4,6,8,15,20])
print(scores)
print(scores.mean(axis = 1))

           0         1         2         3
4   0.040114  0.048426  0.063697  0.025681
6   0.040377  0.048768  0.062001  0.024152
8   0.040795  0.049078  0.060839  0.023562
15  0.041762  0.049810  0.059251  0.023697
20  0.042171  0.050013  0.059145  0.024356
4     0.044480
6     0.043825
8     0.043568
15    0.043630
20    0.043921
dtype: float64


*********

In version 3 (where the update rule includes both match result and frame scores), the best score we got was K=3: 0.049733. In version 4, our update rule is purely based on frame scores. The best we got here is K=8: 0.043568, which is much better than version 3. 

In [7]:
from sklearn.metrics import accuracy_score

tournaments = list(matches.tournament_id.unique())

Ks = [4, 6, 8, 15, 20]
nth = [750, 999, 1040, 1080]

scores2 = np.zeros((5,4))

i=0
for K in Ks:
    j=0
    for n in nth:
        past_tournaments = tournaments[:n]
        current = tournaments[n]
        past_index = matches['tournament_id'].map(lambda x: x in past_tournaments)
        

        matches_past = matches[past_index]
        matches_future = matches[matches['tournament_id'] == current]

        #Update statistics of players
        elo = elo_rating()
        elo.update_stats(matches = matches_past, K_factor = K)

        #Store the actual result in y
        matches_future['result'] = 0  
        y = np.array(matches_future['result'])

        #Making prediction
        X = matches_future[['player1', 'player2', 'best_of']]
        prediction, _,_ = elo.predict(X)

        #Record the accuracy score
        from sklearn.metrics import accuracy_score
        scores2[i,j] = accuracy_score(prediction, y)
        j += 1
    i+=1




/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/688427938.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_future['result'] = 0
/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/688427938.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches_future['result'] = 0
/var/folders/pd/b57yb11s57b55h2t3r3c8q1r0000gn/T/ipykernel_2655/688427938.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

In [8]:
scores2 = pd.DataFrame(scores2, index = [4,6,8,15,20])
print(scores2)
print(scores2.mean(axis = 1))

           0         1      2         3
4   0.722222  0.661417  0.672  0.739437
6   0.730159  0.653543  0.696  0.753521
8   0.714286  0.661417  0.712  0.732394
15  0.698413  0.677165  0.744  0.725352
20  0.706349  0.669291  0.736  0.690141
4     0.698769
6     0.708306
8     0.705024
15    0.711233
20    0.700445
dtype: float64


In version3, the mean score is 0.699 for the four chosen tournaments. This score is very close to what we have here.

Combining both tests, we will set the default K-factor to be 8.

In [9]:
#Print out the ranking using K-factor = 8
elo = elo_rating()
elo.update_stats(matches, K_factor = 8)
stats = elo.stats.sort_values(by = 'elo_rating', ascending = False)
stats.head(10)

,elo_rating,matches_played,matches_won,matches_win_rate,frames_played,frames_won,frames_win_rate
Judd Trump,1622,1534,1106,0.720991,10667,6449,0.604575
Zhao Xintong,1620,375,223,0.594667,2525,1419,0.561980
John Higgins,1602,1933,1363,0.705122,15024,8851,0.589124
Mark Selby,1598,1640,1121,0.683537,11668,6793,0.582191
Yan Bingtao,1574,354,232,0.655367,2479,1406,0.567164
Kyren Wilson,1559,1122,721,0.642602,7218,4082,0.565531
Barry Hawkins,1551,1391,870,0.625449,9636,5396,0.559983
Mark Williams,1542,1835,1201,0.654496,13500,7733,0.572815
Xiao Guodong,1542,814,475,0.583538,5179,2844,0.549141
Ronnie O'Sullivan,1538,1555,1181,0.759486,13448,8192,0.609161
